In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Standardized project root discovery
root = Path.cwd().resolve()
while root != root.parent and not (root / 'README.md').exists():
    root = root.parent
PROJECT_ROOT = root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os
print(os.listdir(PROJECT_ROOT / 'data/processed'))
print(os.listdir(PROJECT_ROOT / 'data/raw'))

['X_features.csv', 'y_facility.csv', 'y_household.csv', 'y_logistic.csv']
['NFHS5_Individual.csv']


In [2]:
import pandas as pd
df = pd.read_csv(PROJECT_ROOT / 'data/raw/NFHS5_Individual.csv')
print(df.columns.tolist())

C:\Users\RABIYA BUSHRA\AppData\Local\Temp\ipykernel_16812\1970966262.py:2: DtypeWarning: Columns (0: s245a, 1: s245b, 2: s245h) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/NFHS5_Individual.csv')


['caseid', 'v001', 'v002', 'v021', 'v024', 'v025', 'v012', 'v013', 'v106', 'v130', 'v131', 'v501', 'v717', 'v190', 'v169a', 'v170', 'v481', 'v157', 'v158', 'v159', 'v743f', 'v466', 'v467b', 'v467c', 'v467d', 'v467e', 'v467g', 'v467h', 'v626a', 's245a', 's245b', 's245h']


In [3]:
print(df['v626a'].value_counts(dropna=False))

v626a
using for limiting                        294315
never had sex                             186301
infecund, menopausal                       68125
no unmet need                              66783
using for spacing                          56089
unmet need for limiting                    27493
unmet need for spacing                     22179
not married and no sex in last 30 days      2670
NaN                                          160
Name: count, dtype: int64


In [4]:
valid_categories = ['no unmet need', 'using for spacing', 'using for limiting',
                     'unmet need for spacing', 'unmet need for limiting']

mask_fp = df['v626a'].isin(valid_categories)
target_unmet_fp = df.loc[mask_fp, 'v626a'].str.contains('unmet need').astype(int)

print('Restricted N for target_unmet_fp:', mask_fp.sum())
print(target_unmet_fp.value_counts())

Restricted N for target_unmet_fp: 466859
v626a
0    350404
1    116455
Name: count, dtype: int64


In [5]:
positive_categories = ['unmet need for spacing', 'unmet need for limiting']

target_unmet_fp = df.loc[mask_fp, 'v626a'].isin(positive_categories).astype(int)

print('Restricted N for target_unmet_fp:', mask_fp.sum())
print(target_unmet_fp.value_counts())

Restricted N for target_unmet_fp: 466859
v626a
0    417187
1     49672
Name: count, dtype: int64


## ⚠️ CRITICAL FINDING: m14 Column is Missing

The NFHS5_Individual.csv extract does **NOT** contain the `m14` column (ANC visit count).

According to Section 2.4 of the Stage2_Implementation_Guide_v2:
> "m14 and v626a were listed in the original cols_needed but need to be confirmed as present in the saved NFHS5_Individual.csv extract — if either was dropped, re-run Step 1 of the Stage 1 v5 load with them included."

**Impact:**
- `target_anc_gap` cannot be built
- Stage 2 will proceed with ONLY `target_unmet_fp` (family planning unmet need)
- This reduces Stage 2 from 2 targets to 1 target

**Recommended Action:**
1. Either: Re-extract from full IAIR7EFL.DTA with m14 included
2. Or: Proceed with single-target Stage 2 and document this limitation

In [11]:
print(df.columns.tolist())

['caseid', 'v001', 'v002', 'v021', 'v024', 'v025', 'v012', 'v013', 'v106', 'v130', 'v131', 'v501', 'v717', 'v190', 'v169a', 'v170', 'v481', 'v157', 'v158', 'v159', 'v743f', 'v466', 'v467b', 'v467c', 'v467d', 'v467e', 'v467g', 'v467h', 'v626a', 's245a', 's245b', 's245h']


In [1]:
import pandas as pd
X = pd.read_csv(PROJECT_ROOT / 'data/processed/X_features.csv')
print(X.shape)
print(X.columns.tolist())

(724115, 56)
['v012', 'media_exposure_index', 'v013_20-24', 'v013_25-29', 'v013_30-34', 'v013_35-39', 'v013_40-44', 'v013_45-49', 'v106_no education', 'v106_primary', 'v106_secondary', 'v130_christian', 'v130_hindu', 'v130_jain', 'v130_jewish', 'v130_muslim', 'v130_no religion', 'v130_other', 'v130_parsi / zoroastrian', 'v130_sikh', "v131_don't know", 'v131_no caste / tribe', 'v131_tribe', 'v501_married', 'v501_never in union  [includes: married gauna not performed]', 'v501_no longer living together/separated', 'v501_widowed', 'v717_clerical', "v717_don't know", 'v717_missing', 'v717_not working', 'v717_other', 'v717_professional / technical / managerial', 'v717_sales', 'v717_services / household and domestic', 'v717_skilled and unskilled manual', 'v190_poorer', 'v190_poorest', 'v190_richer', 'v190_richest', 'v169a_no', 'v169a_yes', 'v170_no', 'v170_yes', 'v481_yes', 'v157_less than once a week', 'v157_not at all', 'v158_less than once a week', 'v158_not at all', 'v159_less than once a

In [2]:
oof_df = pd.read_csv(PROJECT_ROOT / 'data/processed/stage2/oof_barrier_probabilities.csv')
print(oof_df.shape)
print(oof_df.describe())

(724115, 4)
       household_barrier_prob  logistic_barrier_prob  facility_barrier_prob  \
count           724115.000000          724115.000000          724115.000000   
mean                 0.478844               0.482977               0.497819   
std                  0.147943               0.149016               0.109300   
min                  0.016792               0.016937               0.053281   
25%                  0.370945               0.378120               0.431704   
50%                  0.488287               0.498524               0.503505   
75%                  0.593619               0.591880               0.577197   
max                  0.947311               0.953323               0.916686   

       composite_barrier_score  
count            724115.000000  
mean                  0.486547  
std                   0.130525  
min                   0.046865  
25%                   0.395719  
50%                   0.496208  
75%                   0.583405  
max         

In [3]:
import joblib
from sklearn.metrics import roc_auc_score

y_household = pd.read_csv(PROJECT_ROOT / 'data/processed/y_household.csv').squeeze()
y_logistic  = pd.read_csv(PROJECT_ROOT / 'data/processed/y_logistic.csv').squeeze()
y_facility  = pd.read_csv(PROJECT_ROOT / 'data/processed/y_facility.csv').squeeze()

dt_hh = joblib.load(PROJECT_ROOT / 'saved_models/stage1/decision_tree_household.pkl')
dt_lg = joblib.load(PROJECT_ROOT / 'saved_models/stage1/decision_tree_logistic.pkl')
dt_fc = joblib.load(PROJECT_ROOT / 'saved_models/stage1/decision_tree_facility.pkl')

X = pd.read_csv(PROJECT_ROOT / 'data/processed/X_features.csv')

in_sample_hh = dt_hh.predict_proba(X)[:, 1]
in_sample_lg = dt_lg.predict_proba(X)[:, 1]
in_sample_fc = dt_fc.predict_proba(X)[:, 1]

for name, oof_col, in_sample_probs, y_true in [
    ('household', 'household_barrier_prob', in_sample_hh, y_household),
    ('logistic',  'logistic_barrier_prob',  in_sample_lg, y_logistic),
    ('facility',  'facility_barrier_prob',  in_sample_fc, y_facility),
]:
    auc_oof = roc_auc_score(y_true, oof_df[oof_col])
    auc_in_sample = roc_auc_score(y_true, in_sample_probs)
    print(f'{name}: in-sample AUC={auc_in_sample:.4f} | OOF AUC={auc_oof:.4f} | gap={auc_in_sample - auc_oof:+.4f}')

C:\Users\RABIYA BUSHRA\OneDrive\Attachments\Desktop\MajorProject\Implementation\BarrierLens_MP_G25_P48\venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
C:\Users\RABIYA BUSHRA\OneDrive\Attachments\Desktop\MajorProject\Implementation\BarrierLens_MP_G25_P48\venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
C:\Users\RABIYA BUSHRA\OneDrive\Attachments\Desktop\MajorProject\Implementation\BarrierLens_MP_G25_P48\venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(


household: in-sample AUC=0.6504 | OOF AUC=0.6640 | gap=-0.0136
logistic: in-sample AUC=0.6510 | OOF AUC=0.6640 | gap=-0.0129
facility: in-sample AUC=0.5988 | OOF AUC=0.6201 | gap=-0.0214


In [4]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

base_kwargs = dict(max_depth=6, n_estimators=300, learning_rate=0.08,
                    subsample=0.8, colsample_bytree=0.8, tree_method='hist', random_state=42)

X_clean = X.copy()
X_clean.columns = X_clean.columns.astype(str).str.replace(r'[\[\]<]', '', regex=True)

for name, y_true in [('household', y_household), ('logistic', y_logistic), ('facility', y_facility)]:
    neg, pos = (y_true == 0).sum(), (y_true == 1).sum()
    model = XGBClassifier(scale_pos_weight=neg / pos, **base_kwargs)
    model.fit(X_clean, y_true)
    in_sample_probs = model.predict_proba(X_clean)[:, 1]

    auc_in_sample = roc_auc_score(y_true, in_sample_probs)
    auc_oof = roc_auc_score(y_true, oof_df[f'{name}_barrier_prob'])
    print(f'{name}: in-sample AUC={auc_in_sample:.4f} | OOF AUC={auc_oof:.4f} | gap={auc_in_sample - auc_oof:+.4f}')

household: in-sample AUC=0.6749 | OOF AUC=0.6640 | gap=+0.0110
logistic: in-sample AUC=0.6741 | OOF AUC=0.6640 | gap=+0.0101
facility: in-sample AUC=0.6314 | OOF AUC=0.6201 | gap=+0.0113
